In [1]:
import duckdb
from pathlib import Path
con = duckdb.connect('../data/cfpb.duckdb')
con.sql("SELECT COUNT(*) AS rows FROM complaints_scoped").show()

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 533015 │
└────────┘



In [4]:
def run_sql_file(name):
    text = Path(f'../sql/{name}').read_text()
    for stmt in text.split(';'):
        lines = stmt.splitlines()
        body = '\n'.join(l for l in lines if not l.strip().startswith('--')).strip()
        if not body:
            continue
        labels = [l for l in lines if l.strip().startswith('--')]
        print(labels[-1] if labels else '')
        rel = con.sql(stmt)
        if rel is not None:
            display(rel.df())

In [3]:
con.sql("SELECT company_response, COUNT(*) AS n FROM complaints_scoped GROUP BY 1 ORDER BY 2 DESC").show()
con.sql("SELECT timely_response, COUNT(*) AS n FROM complaints_scoped GROUP BY 1 ORDER BY 2 DESC").show()

┌─────────────────────────────────┬────────┐
│        company_response         │   n    │
│             varchar             │ int64  │
├─────────────────────────────────┼────────┤
│ Closed with explanation         │ 426748 │
│ Closed with monetary relief     │  69924 │
│ Closed with non-monetary relief │  35393 │
│ Untimely response               │    815 │
│ In progress                     │    135 │
└─────────────────────────────────┴────────┘

┌─────────────────┬────────┐
│ timely_response │   n    │
│     varchar     │ int64  │
├─────────────────┼────────┤
│ Yes             │ 527053 │
│ No              │   5962 │
└─────────────────┴────────┘



In [5]:
run_sql_file('04_trend.sql')

-- 0. Analysis view: full months only (Aug 2026 is a partial month)


,Count


-- 1. Monthly volume by product with MoM change and 3-month rolling average


,month,product,complaints,complaints_ex_surge,mom_change,mom_pct,rolling_3m
0,2023-09-01,Checking or savings account,3771,3771,<NA>,NaN,3771.0
1,2023-10-01,Checking or savings account,3766,3766,-5,-0.1,3769.0
2,2023-11-01,Checking or savings account,3744,3744,-22,-0.6,3760.0
3,2023-12-01,Checking or savings account,3792,3792,48,1.3,3767.0
4,2024-01-01,Checking or savings account,3993,3993,201,5.3,3843.0
...,...,...,...,...,...,...,...
135,2026-03-01,Prepaid card,460,460,-36,-7.3,533.0
136,2026-04-01,Prepaid card,455,455,-5,-1.1,470.0
137,2026-05-01,Prepaid card,499,499,44,9.7,471.0
138,2026-06-01,Prepaid card,528,528,29,5.8,494.0


In [6]:
run_sql_file('05_top_issues.sql')


-- 1. Top 10 issues per product, ranked by volume


,product,issue,complaints,pct_of_product,rnk
0,Checking or savings account,Managing an account,109814,56.8,1
1,Checking or savings account,Closing an account,26751,13.8,2
2,Checking or savings account,Problem with a lender or other company chargin...,25595,13.2,3
3,Checking or savings account,Opening an account,17052,8.8,4
4,Checking or savings account,Problem caused by your funds being low,14211,7.3,5
5,Credit card,Problem with a purchase shown on your statement,57650,33.1,1
6,Credit card,"Other features, terms, or problems",23750,13.6,2
7,Credit card,Getting a credit card,23623,13.6,3
8,Credit card,Fees or interest,21571,12.4,4
9,Credit card,Closing your account,15169,8.7,5


-- 2. Running share: how quickly the top issues add up to the whole product


,product,issue_rank,issue,complaints,cum_pct
0,Checking or savings account,1,Managing an account,109814,56.8
1,Checking or savings account,2,Closing an account,26751,70.6
2,Checking or savings account,3,Problem with a lender or other company chargin...,25595,83.8
3,Checking or savings account,4,Opening an account,17052,92.7
4,Checking or savings account,5,Problem caused by your funds being low,14211,100.0
5,Credit card,1,Problem with a purchase shown on your statement,57650,33.1
6,Credit card,2,"Other features, terms, or problems",23750,46.7
7,Credit card,3,Getting a credit card,23623,60.2
8,Credit card,4,Fees or interest,21571,72.6
9,Credit card,5,Closing your account,15169,81.3


In [8]:
run_sql_file('06_growth.sql')

-- 1. Fastest growing issues by percent


,product,issue,prior_12m,last_12m,change,growth_pct
0,"Money transfer, virtual currency, or money ser...","Managing, opening, or closing your mobile wall...",1258,2305,1047,83.2
1,"Money transfer, virtual currency, or money ser...",Trouble accessing funds in your mobile or digi...,2199,3927,1728,78.6
2,Credit card,Problem with a purchase shown on your statement,16467,28368,11901,72.3
3,Checking or savings account,Problem with a lender or other company chargin...,7818,12193,4375,56.0
4,"Money transfer, virtual currency, or money ser...",Unauthorized transactions or other transaction...,4279,6330,2051,47.9
5,Credit card,"Advertising and marketing, including promotion...",2703,3903,1200,44.4
6,"Money transfer, virtual currency, or money ser...",Lost or stolen money order,205,290,85,41.5
7,Checking or savings account,Managing an account,35210,49107,13897,39.5
8,Prepaid card,Problem with a purchase or transfer,1574,2184,610,38.8
9,"Money transfer, virtual currency, or money ser...",Money was not available when promised,2047,2740,693,33.9


-- 2. Biggest absolute increases


,product,issue,prior_12m,last_12m,change,growth_pct
0,Checking or savings account,Managing an account,35210,49107,13897,39.5
1,Credit card,Problem with a purchase shown on your statement,16467,28368,11901,72.3
2,Checking or savings account,Problem with a lender or other company chargin...,7818,12193,4375,56.0
3,Checking or savings account,Closing an account,8831,11449,2618,29.6
4,Credit card,"Other features, terms, or problems",7601,10141,2540,33.4
5,"Money transfer, virtual currency, or money ser...",Unauthorized transactions or other transaction...,4279,6330,2051,47.9
6,Credit card,Fees or interest,7383,9171,1788,24.2
7,"Money transfer, virtual currency, or money ser...",Trouble accessing funds in your mobile or digi...,2199,3927,1728,78.6
8,"Money transfer, virtual currency, or money ser...",Fraud or scam,9771,11179,1408,14.4
9,Checking or savings account,Opening an account,5905,7164,1259,21.3


In [9]:
run_sql_file('07_peer_compare.sql')

-- 1. Issue mix gap, all fintech companies


,topic,fintech_pct,bank_pct,gap_pts
0,"Money transfer, virtual currency, or money ser...",15.7,1.8,13.9
1,Credit card | Problem with a purchase shown on...,3.0,15.8,-12.8
2,"Money transfer, virtual currency, or money ser...",10.4,0.5,9.9
3,"Money transfer, virtual currency, or money ser...",12.8,3.9,8.9
4,Checking or savings account | Managing an account,19.2,26.7,-7.5
5,"Money transfer, virtual currency, or money ser...",6.8,0.1,6.8
6,"Credit card | Other features, terms, or problems",0.6,6.7,-6.1
7,Credit card | Getting a credit card,0.7,6.0,-5.3
8,Credit card | Fees or interest,0.2,4.5,-4.3
9,"Money transfer, virtual currency, or money ser...",4.1,0.0,4.1


-- 2. Same comparison without Block (Block is over half of all fintech complaints)


,topic,fintech_pct,bank_pct,gap_pts
0,Credit card | Problem with a purchase shown on...,4.1,15.8,-11.7
1,"Money transfer, virtual currency, or money ser...",8.2,0.5,7.7
2,"Money transfer, virtual currency, or money ser...",7.6,0.1,7.5
3,"Money transfer, virtual currency, or money ser...",11.3,3.9,7.4
4,"Money transfer, virtual currency, or money ser...",8.0,1.8,6.2
5,"Credit card | Other features, terms, or problems",0.8,6.7,-6.0
6,Credit card | Getting a credit card,0.9,6.0,-5.2
7,Credit card | Fees or interest,0.3,4.5,-4.2
8,"Money transfer, virtual currency, or money ser...",4.0,0.0,3.9
9,Credit card | Closing your account,0.4,4.0,-3.6


In [10]:
run_sql_file('08_outcomes.sql')

-- 0. Resolved complaints with outcome flags


,Count


-- 1. Outcomes by product


,product,complaints,relief_pct,untimely_pct,poor_outcome_pct
0,Checking or savings account,193396,19.3,0.7,80.8
1,Credit card,174309,29.7,0.8,70.5
2,"Money transfer, virtual currency, or money ser...",93008,10.2,2.2,89.9
3,Prepaid card,18494,26.5,5.7,73.5


-- 2. Worst five issues per product by poor outcome rate (issues with 500 or more complaints)


,product,issue,complaints,relief_pct,untimely_pct,poor_outcome_pct,poor_rank
0,Checking or savings account,Opening an account,17047,16.6,0.5,83.5,1
1,Checking or savings account,Problem with a lender or other company chargin...,25589,18.8,0.8,81.3,2
2,Checking or savings account,Managing an account,109804,19.4,0.7,80.7,3
3,Checking or savings account,Closing an account,26747,20.4,0.9,79.7,4
4,Checking or savings account,Problem caused by your funds being low,14209,20.6,0.4,79.5,5
5,Credit card,Trouble using your card,8099,15.7,0.9,84.4,1
6,Credit card,Struggling to pay your bill,3036,18.1,0.7,81.9,2
7,Credit card,Closing your account,15169,20.1,0.8,79.9,3
8,Credit card,"Advertising and marketing, including promotion...",8650,26.5,0.8,73.6,4
9,Credit card,"Other features, terms, or problems",23748,27.6,0.8,72.4,5


-- 3. Fintech vs Bank


,company_group,complaints,relief_pct,untimely_pct,poor_outcome_pct
0,Fintech,91698,10.7,1.6,89.3
1,Bank,180409,27.7,0.2,72.4


-- 4. Each peer company with 1000 or more complaints


,company_group,company,complaints,relief_pct,untimely_pct,poor_outcome_pct
0,Bank,JPMORGAN CHASE & CO.,44703,15.8,0.0,84.2
1,Bank,WELLS FARGO & COMPANY,33976,18.9,0.0,81.1
2,Bank,CAPITAL ONE FINANCIAL CORPORATION,33796,20.8,0.0,79.2
3,Bank,"BANK OF AMERICA, NATIONAL ASSOCIATION",34754,41.0,0.8,59.5
4,Bank,"CITIBANK, N.A.",33180,46.0,0.0,54.0
5,Fintech,ROBINHOOD MARKETS INC.,3866,0.0,9.7,100.0
6,Fintech,"Block, Inc.",37583,0.1,2.4,99.9
7,Fintech,TransferWise Ltd,1367,9.0,7.0,92.2
8,Fintech,"SOFI TECHNOLOGIES, INC.",5332,8.3,0.1,91.7
9,Fintech,Chime Financial Inc,19228,9.2,0.0,90.8


In [11]:
run_sql_file('09_priority.sql')

-- 1. Metrics per topic


,Count
0,28


-- 2. Fixability ratings from 0 to 1, left empty for now so every topic defaults to 0.5


,Count


-- 3. Score and rank


,Count
0,28


-- 4. Top 25 candidates


,priority_rank,product,issue,complaints,growth_pct,poor_outcome_pct,fixability,score
0,1,Checking or savings account,Managing an account,109804,39.4,80.7,0.5,0.696
1,2,Checking or savings account,Problem with a lender or other company chargin...,25589,55.9,81.3,0.5,0.689
2,3,"Money transfer, virtual currency, or money ser...",Fraud or scam,26147,14.4,91.1,0.5,0.672
3,4,"Money transfer, virtual currency, or money ser...",Unauthorized transactions or other transaction...,12007,47.9,85.6,0.5,0.654
4,5,Credit card,Problem with a purchase shown on your statement,57647,72.3,66.4,0.5,0.639
5,6,"Money transfer, virtual currency, or money ser...","Managing, opening, or closing your mobile wall...",4258,83.1,87.6,0.5,0.631
6,7,Checking or savings account,Closing an account,26747,29.6,79.7,0.5,0.609
7,8,"Money transfer, virtual currency, or money ser...",Other transaction problem,25369,-52.9,93.7,0.5,0.594
8,9,"Money transfer, virtual currency, or money ser...",Trouble accessing funds in your mobile or digi...,7285,78.5,83.4,0.5,0.591
9,10,Credit card,Trouble using your card,8099,33.5,84.4,0.5,0.556


In [12]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 300)
print(con.sql("""
SELECT date_trunc('week', date_received) AS week_start,
       COUNT(*) AS combo_rows,
       COUNT(*) FILTER (WHERE is_surge_event) AS flagged
FROM complaints_scoped
WHERE company IN ('Block, Inc.', 'Early Warning Services, LLC',
                  'NAVY FEDERAL CREDIT UNION', 'CAPITAL ONE FINANCIAL CORPORATION')
  AND issue IN ('Other transaction problem', 'Problem caused by your funds being low',
                'Managing an account')
  AND date_received BETWEEN DATE '2024-11-01' AND DATE '2025-04-30'
GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

week_start  combo_rows  flagged
2024-10-28          23        0
2024-11-04         214        0
2024-11-11         604        0
2024-11-18         119        0
2024-11-25          75        0
2024-12-02         100        0
2024-12-09          84        0
2024-12-16         127        0
2024-12-23         100        0
2024-12-30         108        0
2025-01-06          87        0
2025-01-13       41028    40940
2025-01-20       10662     4398
2025-01-27        2828        0
2025-02-03        1764        0
2025-02-10        1176        0
2025-02-17         537        0
2025-02-24         548        0
2025-03-03         319        0
2025-03-10         321        0
2025-03-17         448        0
2025-03-24         178        0
2025-03-31         169        0
2025-04-07         205        0
2025-04-14         211        0
2025-04-21         263        0
2025-04-28          79        0


In [13]:
print('--- Overall growth baseline')
print(con.sql("""
SELECT COUNT(*) FILTER (WHERE date_received >= DATE '2025-08-01') AS last_12m,
       COUNT(*) FILTER (WHERE date_received >= DATE '2024-08-01'
                          AND date_received <  DATE '2025-08-01') AS prior_12m
FROM analysis
WHERE NOT is_surge_event AND date_received >= DATE '2024-08-01'
""").df().to_string(index=False))

print('--- Top 3 companies behind each candidate issue')
print(con.sql("""
WITH t AS (
  SELECT product, issue, company,
         COUNT(*) FILTER (WHERE date_received >= DATE '2025-08-01') AS last_12m,
         COUNT(*) FILTER (WHERE date_received >= DATE '2024-08-01'
                            AND date_received <  DATE '2025-08-01') AS prior_12m
  FROM analysis
  WHERE NOT is_surge_event AND date_received >= DATE '2024-08-01'
    AND issue IN ('Managing an account',
                  'Problem with a lender or other company charging your account',
                  'Fraud or scam',
                  'Unauthorized transactions or other transaction problem',
                  'Problem with a purchase shown on your statement',
                  'Managing, opening, or closing your mobile wallet account',
                  'Trouble accessing funds in your mobile or digital wallet')
  GROUP BY 1, 2, 3
),
r AS (
  SELECT *, SUM(last_12m) OVER (PARTITION BY product, issue) AS issue_last,
         ROW_NUMBER() OVER (PARTITION BY product, issue ORDER BY last_12m DESC) AS rn
  FROM t
)
SELECT product, issue, company, prior_12m, last_12m,
       ROUND(100.0 * last_12m / issue_last, 1) AS share_of_issue_pct
FROM r WHERE rn <= 3
ORDER BY product, issue, rn
""").df().to_string(index=False))

print('--- Sub-issues of the two biggest buckets')
print(con.sql("""
SELECT product, issue, sub_issue,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-08-01') AS last_12m,
       COUNT(*) FILTER (WHERE date_received >= DATE '2024-08-01'
                          AND date_received <  DATE '2025-08-01') AS prior_12m
FROM analysis
WHERE NOT is_surge_event AND date_received >= DATE '2024-08-01'
  AND ((product = 'Checking or savings account' AND issue = 'Managing an account')
    OR (product = 'Credit card' AND issue = 'Problem with a purchase shown on your statement'))
GROUP BY 1, 2, 3
ORDER BY last_12m DESC
""").df().to_string(index=False))

print('--- Block response labels')
print(con.sql("""
SELECT company_response, COUNT(*) AS n
FROM outcomes WHERE company = 'Block, Inc.'
GROUP BY 1 ORDER BY 2 DESC
""").df().to_string(index=False))

print('--- Fintech relief rate without Block')
print(con.sql("""
SELECT COUNT(*) AS complaints, ROUND(100.0 * AVG(o.got_relief::INT), 1) AS relief_pct
FROM outcomes o JOIN company_groups cg ON o.company = cg.company
WHERE cg.company_group = 'Fintech' AND o.company <> 'Block, Inc.'
""").df().to_string(index=False))

--- Overall growth baseline
 last_12m  prior_12m
   205052     167357
--- Top 3 companies behind each candidate issue
                                           product                                                        issue                               company  prior_12m  last_12m  share_of_issue_pct
                       Checking or savings account                                          Managing an account                 WELLS FARGO & COMPANY       4202      5693                11.6
                       Checking or savings account                                          Managing an account                  JPMORGAN CHASE & CO.       4488      5376                10.9
                       Checking or savings account                                          Managing an account BANK OF AMERICA, NATIONAL ASSOCIATION       3272      5034                10.3
                       Checking or savings account Problem with a lender or other company charging your account       

In [14]:
print(con.sql("""
SELECT strftime(date_trunc('month', date_received), '%Y-%m') AS month,
       COUNT(*) FILTER (WHERE company = 'Block, Inc.') AS block,
       COUNT(*) FILTER (WHERE company = 'Early Warning Services, LLC') AS ews,
       COUNT(*) FILTER (WHERE company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')) AS everyone_else
FROM analysis
GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

  month  block   ews  everyone_else
2023-09    256    59           8366
2023-10    270    39           9291
2023-11    219    78           8700
2023-12    245    63           9123
2024-01    224    42           9694
2024-02    241    50           9313
2024-03    240    49          10187
2024-04    270    30           9918
2024-05    253    48          10115
2024-06    242    61           9012
2024-07    273    58           9836
2024-08    359   111          10488
2024-09    255    64           9652
2024-10    333    92          10394
2024-11    289    87          10527
2024-12    358   112          11114
2025-01  31221 16722          26604
2025-02   3822  1353          11129
2025-03   1710   276          10958
2025-04   1295   146          11151
2025-05   1335   113          12028
2025-06   1168   152          12090
2025-07   1239   177          13771
2025-08   1214   196          13515
2025-09    986   153          13228
2025-10   1191   976          13803
2025-11   1046   106        

In [15]:
print(con.sql("""
WITH t AS (
  SELECT CASE product WHEN 'Checking or savings account' THEN 'Checking'
                      WHEN 'Money transfer, virtual currency, or money service' THEN 'Money transfer'
                      ELSE product END AS prod,
         SUBSTR(issue, 1, 40) AS issue_short,
         SUBSTR(company, 1, 22) AS company_short,
         COUNT(*) AS last_12m
  FROM analysis
  WHERE NOT is_surge_event AND date_received >= DATE '2025-08-01'
    AND issue IN ('Managing an account',
                  'Problem with a lender or other company charging your account',
                  'Fraud or scam',
                  'Unauthorized transactions or other transaction problem',
                  'Problem with a purchase shown on your statement',
                  'Managing, opening, or closing your mobile wallet account',
                  'Trouble accessing funds in your mobile or digital wallet')
  GROUP BY 1, 2, 3
),
r AS (
  SELECT *, SUM(last_12m) OVER (PARTITION BY prod, issue_short) AS issue_total,
         ROW_NUMBER() OVER (PARTITION BY prod, issue_short ORDER BY last_12m DESC) AS rn
  FROM t
)
SELECT prod, issue_short, company_short, last_12m,
       ROUND(100.0 * last_12m / issue_total, 1) AS share_pct
FROM r WHERE rn <= 3
ORDER BY prod, issue_short, rn
""").df().to_string(index=False))

          prod                              issue_short          company_short  last_12m  share_pct
      Checking                      Managing an account  WELLS FARGO & COMPANY      5693       11.6
      Checking                      Managing an account   JPMORGAN CHASE & CO.      5376       10.9
      Checking                      Managing an account BANK OF AMERICA, NATIO      5034       10.3
      Checking Problem with a lender or other company c    Chime Financial Inc      2047       16.8
      Checking Problem with a lender or other company c  WELLS FARGO & COMPANY      1522       12.5
      Checking Problem with a lender or other company c   JPMORGAN CHASE & CO.      1413       11.6
   Credit card Problem with a purchase shown on your st         CITIBANK, N.A.      4283       15.1
   Credit card Problem with a purchase shown on your st CAPITAL ONE FINANCIAL       3686       13.0
   Credit card Problem with a purchase shown on your st BANK OF AMERICA, NATIO      2570        9.1


In [16]:
print(con.sql("""
SELECT CASE product WHEN 'Checking or savings account' THEN 'Checking' ELSE product END AS prod,
       SUBSTR(sub_issue, 1, 70) AS sub_issue_short,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-08-01') AS last_12m,
       COUNT(*) FILTER (WHERE date_received >= DATE '2024-08-01'
                          AND date_received <  DATE '2025-08-01') AS prior_12m
FROM analysis
WHERE NOT is_surge_event AND date_received >= DATE '2024-08-01'
  AND ((product = 'Checking or savings account' AND issue = 'Managing an account')
    OR (product = 'Credit card' AND issue = 'Problem with a purchase shown on your statement'))
GROUP BY 1, 2
ORDER BY prod, last_12m DESC
""").df().to_string(index=False))

       prod                                                        sub_issue_short  last_12m  prior_12m
   Checking                                               Deposits and withdrawals     16610      12613
   Checking                                      Problem using a debit or ATM card     13064       8085
   Checking                           Funds not handled or disbursed as instructed      5758       4174
   Checking                                                         Banking errors      4079       3264
   Checking                                              Problem accessing account      3883       2956
   Checking                                   Problem making or receiving payments      3382       2158
   Checking                                                        Cashing a check      1107        907
   Checking                                                            Fee problem       806        666
   Checking                                                Depos

In [17]:
from pathlib import Path

scope_sql = """-- 03_scope.sql
-- Purpose: flag complaints to leave out of trend analysis and build the scoped table
-- Flags: is_bureau (credit bureaus), is_credit_report_issue, is_surge_event
-- is_surge_event covers two January 2025 anomalies:
--   1. Block and Early Warning Services (Zelle), Jan 13 to Aug 31 2025, a complaint campaign
--      driven by social media influencers as reported by Consumer Reports
--   2. Any other company whose Jan 13 to Jan 31 2025 volume was over 500 and over 3x its
--      December 2024 volume, which catches the Capital One vendor outage
-- Flagged rows are kept in the table, not deleted

DROP VIEW IF EXISTS outcomes;
DROP VIEW IF EXISTS analysis;

CREATE OR REPLACE TABLE spike_companies AS
SELECT company
FROM complaints
WHERE date_received BETWEEN DATE '2024-12-01' AND DATE '2025-01-31'
GROUP BY company
HAVING COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2025-01-13' AND DATE '2025-01-31') > 500
   AND COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2025-01-13' AND DATE '2025-01-31')
     > 3 * COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2024-12-01' AND DATE '2024-12-31');

CREATE OR REPLACE TABLE complaints_flagged AS
SELECT *,
  company IN ('EQUIFAX, INC.',
              'TRANSUNION INTERMEDIATE HOLDINGS, INC.',
              'Experian Information Solutions Inc.') AS is_bureau,
  (issue ILIKE '%report%' OR issue ILIKE '%credit score%'
   OR issue ILIKE '%investigation%' OR issue ILIKE '%fraud alert%'
   OR issue ILIKE '%identity theft%' OR issue ILIKE '%monitoring%') AS is_credit_report_issue
FROM complaints;

CREATE OR REPLACE TABLE complaints_scoped AS
SELECT * EXCLUDE (is_bureau, is_credit_report_issue),
  (
    (company IN ('Block, Inc.', 'Early Warning Services, LLC')
       AND date_received BETWEEN DATE '2025-01-13' AND DATE '2025-08-31')
    OR
    (company IN (SELECT company FROM spike_companies)
       AND date_received BETWEEN DATE '2025-01-13' AND DATE '2025-01-31')
  ) AS is_surge_event
FROM complaints_flagged
WHERE NOT is_bureau AND NOT is_credit_report_issue;
"""
Path('../sql/03_scope.sql').write_text(scope_sql)

fix_sql = """-- 10_fixability.sql
-- Purpose: analyst ratings from 0 to 1 of how much a neobank's own product, policy and operations can change each issue
-- 0.8 mostly in the company's control, 0.5 to 0.7 mixed, 0.3 to 0.4 pricing or vague buckets
-- These are judgments, not data, and the memo will say so

DELETE FROM fixability;

INSERT INTO fixability VALUES
 ('Checking or savings account', 'Managing an account', 0.7),
 ('Checking or savings account', 'Problem with a lender or other company charging your account', 0.7),
 ('Checking or savings account', 'Closing an account', 0.8),
 ('Checking or savings account', 'Opening an account', 0.7),
 ('Checking or savings account', 'Problem caused by your funds being low', 0.5),
 ('Money transfer, virtual currency, or money service', 'Fraud or scam', 0.5),
 ('Money transfer, virtual currency, or money service', 'Unauthorized transactions or other transaction problem', 0.6),
 ('Money transfer, virtual currency, or money service', 'Managing, opening, or closing your mobile wallet account', 0.8),
 ('Money transfer, virtual currency, or money service', 'Trouble accessing funds in your mobile or digital wallet', 0.8),
 ('Money transfer, virtual currency, or money service', 'Other transaction problem', 0.4),
 ('Money transfer, virtual currency, or money service', 'Money was not available when promised', 0.6),
 ('Money transfer, virtual currency, or money service', 'Problem with customer service', 0.8),
 ('Money transfer, virtual currency, or money service', 'Other service problem', 0.3),
 ('Money transfer, virtual currency, or money service', 'Confusing or missing disclosures', 0.8),
 ('Money transfer, virtual currency, or money service', 'Unexpected or other fees', 0.4),
 ('Credit card', 'Problem with a purchase shown on your statement', 0.5),
 ('Credit card', 'Trouble using your card', 0.6),
 ('Credit card', 'Other features, terms, or problems', 0.3),
 ('Credit card', 'Advertising and marketing, including promotional offers', 0.7),
 ('Credit card', 'Closing your account', 0.7),
 ('Credit card', 'Fees or interest', 0.3),
 ('Credit card', 'Getting a credit card', 0.5),
 ('Credit card', 'Struggling to pay your bill', 0.5),
 ('Credit card', 'Problem when making payments', 0.6),
 ('Prepaid card', 'Problem with a purchase or transfer', 0.5);
"""
Path('../sql/10_fixability.sql').write_text(fix_sql)
print('written')

written


In [18]:
def run_quiet(name):
    text = Path(f'../sql/{name}').read_text()
    for stmt in text.split(';'):
        body = '\n'.join(l for l in stmt.splitlines() if not l.strip().startswith('--')).strip()
        if body:
            con.sql(stmt)

# Version A: all companies, flagged rows removed
for f in ['03_scope.sql', '04_trend.sql', '08_outcomes.sql', '10_fixability.sql', '09_priority.sql']:
    run_quiet(f)
con.sql("CREATE OR REPLACE TABLE priority_all AS SELECT * FROM priority")

# Version B: also drop Block and Early Warning Services completely
con.sql("""CREATE OR REPLACE VIEW analysis AS
           SELECT * FROM complaints_scoped
           WHERE date_received < DATE '2026-08-01'
             AND company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')""")
for f in ['08_outcomes.sql', '09_priority.sql']:
    run_quiet(f)
con.sql("CREATE OR REPLACE TABLE priority_xb AS SELECT * FROM priority")

# Restore version A views
for f in ['04_trend.sql', '08_outcomes.sql']:
    run_quiet(f)

print('--- scope check')
print(con.sql("""SELECT COUNT(*) AS rows, COUNT(*) FILTER (WHERE is_surge_event) AS surge_rows
                 FROM complaints_scoped""").df().to_string(index=False))
print('--- other companies flagged')
print(con.sql("""SELECT company, COUNT(*) AS flagged_rows FROM complaints_scoped
                 WHERE is_surge_event AND company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')
                 GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").df().to_string(index=False))
print('--- monthly volume kept, Nov 2024 to Apr 2025')
print(con.sql("""SELECT strftime(date_trunc('month', date_received), '%Y-%m') AS month,
                        COUNT(*) AS all_rows, COUNT(*) FILTER (WHERE NOT is_surge_event) AS kept
                 FROM complaints_scoped
                 WHERE date_received BETWEEN DATE '2024-11-01' AND DATE '2025-04-30'
                 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('--- issues with no fixability rating')
print(con.sql("""SELECT DISTINCT issue FROM priority_all
                 WHERE issue NOT IN (SELECT issue FROM fixability)""").df().to_string(index=False))
print('--- ranking, all companies vs without Block and Zelle')
print(con.sql("""
SELECT a.priority_rank AS rank_all, x.priority_rank AS rank_xb,
       CASE a.product WHEN 'Checking or savings account' THEN 'Checking'
            WHEN 'Money transfer, virtual currency, or money service' THEN 'Money transfer'
            ELSE a.product END AS prod,
       SUBSTR(a.issue, 1, 50) AS issue_short,
       a.complaints AS n_all, a.growth_pct AS growth_all, x.growth_pct AS growth_xb,
       a.poor_outcome_pct AS poor_all, a.fixability
FROM priority_all a
LEFT JOIN priority_xb x ON a.product = x.product AND a.issue = x.issue
ORDER BY a.priority_rank LIMIT 20
""").df().to_string(index=False))

--- scope check
  rows  surge_rows
533015       74909
--- other companies flagged
                          company  flagged_rows
        NAVY FEDERAL CREDIT UNION          7522
CAPITAL ONE FINANCIAL CORPORATION          5451
--- monthly volume kept, Nov 2024 to Apr 2025
  month  all_rows  kept
2024-11     10903 10903
2024-12     11584 11584
2025-01     74547 13834
2025-02     16304 11129
2025-03     12944 10958
2025-04     12592 11151
--- issues with no fixability rating
                                       issue
Problem getting a card or closing an account
                      Trouble using the card
--- ranking, all companies vs without Block and Zelle
 rank_all  rank_xb           prod                                        issue_short  n_all  growth_all  growth_xb  poor_all  fixability
        1        3 Money transfer                                      Fraud or scam  22763        65.1       31.7      89.8         0.5
        2        1       Checking Problem with a lender or o

In [19]:
from pathlib import Path

priority_sql = """-- 09_priority.sql
-- Purpose: rank product and issue topics to pick the three a neobank should fix first
-- Score = 0.30 volume + 0.25 growth + 0.25 poor outcome + 0.20 fixability, each scaled 0 to 1
-- Volume, growth and poor outcome use percentile rank across candidates, so one outlier cannot dominate
-- Growth compares Sep 2025 to Jul 2026 with Sep 2023 to Jul 2024, the same 11 calendar months two years apart
-- Neither window touches the Jan 2025 surge or the Block and Zelle complaint campaign, so no flag distorts the baseline
-- Candidates have 1000 or more complaints and 200 or more in the earlier window

-- 1. Metrics per topic
CREATE OR REPLACE TABLE topic_metrics AS
SELECT product, issue,
       COUNT(*) AS complaints,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
       COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m,
       ROUND(100.0 * AVG(got_relief::INT), 1) AS relief_pct,
       ROUND(100.0 * AVG(was_untimely::INT), 1) AS untimely_pct,
       ROUND(100.0 * AVG(poor_outcome::INT), 1) AS poor_outcome_pct
FROM outcomes
GROUP BY product, issue
HAVING COUNT(*) >= 1000
   AND COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') >= 200;

-- 2. Fixability ratings table
CREATE TABLE IF NOT EXISTS fixability (product VARCHAR, issue VARCHAR, fixability DOUBLE);

-- 3. Score and rank
CREATE OR REPLACE TABLE priority AS
SELECT *, RANK() OVER (ORDER BY score DESC) AS priority_rank
FROM (
  SELECT m.product, m.issue, m.complaints, m.prior_11m, m.last_11m,
         ROUND(100.0 * (m.last_11m - m.prior_11m) / m.prior_11m, 1) AS growth_pct,
         m.relief_pct, m.untimely_pct, m.poor_outcome_pct,
         COALESCE(f.fixability, 0.5) AS fixability,
         ROUND(0.30 * PERCENT_RANK() OVER (ORDER BY m.complaints)
             + 0.25 * PERCENT_RANK() OVER (ORDER BY (m.last_11m - m.prior_11m) * 1.0 / m.prior_11m)
             + 0.25 * PERCENT_RANK() OVER (ORDER BY m.poor_outcome_pct)
             + 0.20 * COALESCE(f.fixability, 0.5), 3) AS score
  FROM topic_metrics m
  LEFT JOIN fixability f ON m.product = f.product AND m.issue = f.issue
);

-- 4. Top 25 candidates
SELECT priority_rank, product, issue, complaints, growth_pct, poor_outcome_pct, fixability, score
FROM priority
ORDER BY priority_rank
LIMIT 25;
"""
Path('../sql/09_priority.sql').write_text(priority_sql)

fix_path = Path('../sql/10_fixability.sql')
fix_text = fix_path.read_text()
fix_text = fix_text.replace(
    " ('Prepaid card', 'Problem with a purchase or transfer', 0.5);",
    " ('Prepaid card', 'Problem with a purchase or transfer', 0.5),\n"
    " ('Prepaid card', 'Trouble using the card', 0.6),\n"
    " ('Prepaid card', 'Problem getting a card or closing an account', 0.7),\n"
    " ('Prepaid card', 'Unexpected or other fees', 0.4);")
assert 'Trouble using the card' in fix_text
fix_path.write_text(fix_text)
print('written')

written


In [20]:
def run_quiet(name):
    text = Path(f'../sql/{name}').read_text()
    for stmt in text.split(';'):
        body = '\n'.join(l for l in stmt.splitlines() if not l.strip().startswith('--')).strip()
        if body:
            con.sql(stmt)

def baseline(label):
    print('--- overall growth, ' + label)
    print(con.sql("""
    SELECT COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
           COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m,
           ROUND(100.0 * (COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01')
             - COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01'))
             / COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01'), 1) AS growth_pct
    FROM analysis""").df().to_string(index=False))

con.sql("CREATE TABLE IF NOT EXISTS fixability (product VARCHAR, issue VARCHAR, fixability DOUBLE)")

# Version A: all companies, flagged rows removed
for f in ['04_trend.sql', '08_outcomes.sql', '10_fixability.sql', '09_priority.sql']:
    run_quiet(f)
con.sql("CREATE OR REPLACE TABLE priority_all AS SELECT * FROM priority")
baseline('all companies')

# Version B: also drop Block and Early Warning Services completely
con.sql("""CREATE OR REPLACE VIEW analysis AS
           SELECT * FROM complaints_scoped
           WHERE date_received < DATE '2026-08-01'
             AND company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')""")
for f in ['08_outcomes.sql', '09_priority.sql']:
    run_quiet(f)
con.sql("CREATE OR REPLACE TABLE priority_xb AS SELECT * FROM priority")
baseline('without Block and Early Warning Services')

# Restore version A views
for f in ['04_trend.sql', '08_outcomes.sql']:
    run_quiet(f)

print('--- topics with no fixability rating (product + issue)')
print(con.sql("""SELECT a.product, SUBSTR(a.issue, 1, 50) AS issue_short FROM priority_all a
                 LEFT JOIN fixability f ON a.product = f.product AND a.issue = f.issue
                 WHERE f.issue IS NULL""").df().to_string(index=False))

print('--- January 2025 kept rows, biggest increases over December 2024')
print(con.sql("""SELECT SUBSTR(company, 1, 30) AS company_short,
       COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2024-12-01' AND DATE '2024-12-31') AS dec_2024,
       COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2025-01-01' AND DATE '2025-01-31') AS jan_2025
FROM complaints_scoped WHERE NOT is_surge_event
  AND date_received BETWEEN DATE '2024-12-01' AND DATE '2025-01-31'
GROUP BY 1 ORDER BY jan_2025 - dec_2024 DESC LIMIT 8""").df().to_string(index=False))

print('--- ranking, all companies vs without Block and Zelle')
print(con.sql("""
SELECT a.priority_rank AS rank_all, x.priority_rank AS rank_xb,
       CASE a.product WHEN 'Checking or savings account' THEN 'Checking'
            WHEN 'Money transfer, virtual currency, or money service' THEN 'Money transfer'
            ELSE a.product END AS prod,
       SUBSTR(a.issue, 1, 50) AS issue_short,
       a.complaints AS n_all, x.complaints AS n_xb,
       a.growth_pct AS growth_all, x.growth_pct AS growth_xb,
       a.poor_outcome_pct AS poor_all, a.fixability
FROM priority_all a
LEFT JOIN priority_xb x ON a.product = x.product AND a.issue = x.issue
ORDER BY a.priority_rank LIMIT 20
""").df().to_string(index=False))

--- overall growth, all companies
 last_11m  prior_11m  growth_pct
   190127     106865        77.9
--- overall growth, without Block and Early Warning Services
 last_11m  prior_11m  growth_pct
   174122     103555        68.1
--- topics with no fixability rating (product + issue)
Empty DataFrame
Columns: [product, issue_short]
Index: []
--- January 2025 kept rows, biggest increases over December 2024
                 company_short  dec_2024  jan_2025
          JPMORGAN CHASE & CO.      1157      1897
BANK OF AMERICA, NATIONAL ASSO       824      1323
         WELLS FARGO & COMPANY       895      1338
                CITIBANK, N.A.       898      1101
           SYNCHRONY FINANCIAL       427       522
           Chime Financial Inc       301       386
      AMERICAN EXPRESS COMPANY       581       664
                 DISCOVER BANK       212       286
--- ranking, all companies vs without Block and Zelle
 rank_all  rank_xb           prod                                        issue_sho

In [21]:
from pathlib import Path

def run_quiet(name):
    text = Path(f'../sql/{name}').read_text()
    for stmt in text.split(';'):
        body = '\n'.join(l for l in stmt.splitlines() if not l.strip().startswith('--')).strip()
        if body:
            con.sql(stmt)

sens_sql = """-- 11_sensitivity.sql
-- Purpose: test whether the top-ranked topics hold up when the weights and the company set change
-- Company sets: all companies (priority_all) and without Block and Early Warning Services (priority_xb)
-- Six weight sets x two company sets = 12 scenarios
-- Weights are volume, growth, poor outcome, fixability

CREATE OR REPLACE TABLE weight_sets AS
SELECT * FROM (VALUES
  ('base',          0.30, 0.25, 0.25, 0.20),
  ('equal',         0.25, 0.25, 0.25, 0.25),
  ('no_fixability', 0.40, 0.30, 0.30, 0.00),
  ('volume_heavy',  0.50, 0.20, 0.20, 0.10),
  ('growth_heavy',  0.20, 0.45, 0.20, 0.15),
  ('outcome_heavy', 0.20, 0.20, 0.45, 0.15)
) AS t(weight_set, w_vol, w_growth, w_out, w_fix);

CREATE OR REPLACE TABLE scenario_ranks AS
WITH pooled AS (
  SELECT 'all' AS company_set, * FROM priority_all
  UNION ALL
  SELECT 'ex_block_zelle' AS company_set, * FROM priority_xb
),
pr AS (
  SELECT company_set, product, issue, fixability,
         PERCENT_RANK() OVER (PARTITION BY company_set ORDER BY complaints) AS p_vol,
         PERCENT_RANK() OVER (PARTITION BY company_set ORDER BY growth_pct) AS p_growth,
         PERCENT_RANK() OVER (PARTITION BY company_set ORDER BY poor_outcome_pct) AS p_out
  FROM pooled
),
scored AS (
  SELECT pr.company_set, w.weight_set, pr.product, pr.issue,
         w.w_vol * p_vol + w.w_growth * p_growth + w.w_out * p_out + w.w_fix * fixability AS score
  FROM pr CROSS JOIN weight_sets w
)
SELECT *, RANK() OVER (PARTITION BY company_set, weight_set ORDER BY score DESC) AS rnk
FROM scored;
"""
Path('../sql/11_sensitivity.sql').write_text(sens_sql)
run_quiet('11_sensitivity.sql')

print('--- topics ranked by how often they land in the top 5 (12 scenarios)')
print(con.sql("""
SELECT CASE product WHEN 'Checking or savings account' THEN 'Checking'
                    WHEN 'Money transfer, virtual currency, or money service' THEN 'Money transfer'
                    ELSE product END AS prod,
       SUBSTR(issue, 1, 50) AS issue_short,
       COUNT(*) FILTER (WHERE rnk <= 3) AS times_top3,
       COUNT(*) FILTER (WHERE rnk <= 5) AS times_top5,
       MIN(rnk) AS best_rank, MAX(rnk) AS worst_rank, COUNT(*) AS scenarios
FROM scenario_ranks
GROUP BY product, issue
ORDER BY times_top5 DESC, times_top3 DESC, worst_rank
LIMIT 12
""").df().to_string(index=False))

--- topics ranked by how often they land in the top 5 (12 scenarios)
          prod                                        issue_short  times_top3  times_top5  best_rank  worst_rank  scenarios
Money transfer                          Other transaction problem          10          11          1           6         12
Money transfer                                      Fraud or scam           7          11          1          11         12
      Checking Problem with a lender or other company charging yo           9          10          1           7         12
      Checking                                Managing an account           4           7          2          11         12
Money transfer                   Confusing or missing disclosures           3           5          2          18         12
   Credit card    Problem with a purchase shown on your statement           1           5          3          19         12
Money transfer Managing, opening, or closing your mobile wallet

In [22]:
print('--- sub-issues: checking issues and the credit card purchase issue (two-year growth)')
print(con.sql("""
SELECT SUBSTR(issue, 1, 28) AS issue_short, SUBSTR(sub_issue, 1, 60) AS sub_issue_short,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
       COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m,
       ROUND(100.0 * (COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01')
         - COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01'))
         / NULLIF(COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01'), 0), 1) AS growth_pct
FROM analysis
WHERE NOT is_surge_event
  AND ((product = 'Checking or savings account'
        AND issue IN ('Managing an account', 'Problem with a lender or other company charging your account'))
    OR (product = 'Credit card' AND issue = 'Problem with a purchase shown on your statement'))
GROUP BY 1, 2
HAVING COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') >= 500
ORDER BY issue_short, last_11m DESC
""").df().to_string(index=False))

print('--- money transfer issues by sub-product')
print(con.sql("""
SELECT SUBSTR(issue, 1, 36) AS issue_short, SUBSTR(sub_product, 1, 30) AS sub_product_short,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
       COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m
FROM analysis
WHERE NOT is_surge_event AND product = 'Money transfer, virtual currency, or money service'
  AND issue IN ('Fraud or scam', 'Other transaction problem',
                'Unauthorized transactions or other transaction problem',
                'Managing, opening, or closing your mobile wallet account',
                'Trouble accessing funds in your mobile or digital wallet')
GROUP BY 1, 2
HAVING COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') >= 300
ORDER BY issue_short, last_11m DESC
""").df().to_string(index=False))

--- sub-issues: checking issues and the credit card purchase issue (two-year growth)
                 issue_short                                              sub_issue_short  last_11m  prior_11m  growth_pct
         Managing an account                                     Deposits and withdrawals     15328       9270        65.4
         Managing an account                            Problem using a debit or ATM card     12244       5599       118.7
         Managing an account                 Funds not handled or disbursed as instructed      5320       2970        79.1
         Managing an account                                               Banking errors      3736       2563        45.8
         Managing an account                                    Problem accessing account      3557       2101        69.3
         Managing an account                         Problem making or receiving payments      3162       1341       135.8
         Managing an account                          

In [23]:
from pathlib import Path

modes_sql = """-- 12_failure_modes.sql
-- Purpose: regroup issues by failure mode instead of product label, so one operational failure
-- that the CFPB form splits across Credit card, Checking, Prepaid and Money transfer shows up as one number
-- The grouping is an analyst judgment, shown in full in the CASE below
-- Windows: last 11m is Sep 2025 to Jul 2026, prior 11m is Sep 2023 to Jul 2024, two years apart
-- Surge-flagged rows are already out of the outcomes view

CREATE OR REPLACE VIEW modes AS
SELECT o.*,
  CASE
    WHEN product = 'Credit card' AND issue = 'Problem with a purchase shown on your statement'
      THEN 'Disputes and unauthorized transactions'
    WHEN product = 'Prepaid card' AND issue = 'Problem with a purchase or transfer'
      THEN 'Disputes and unauthorized transactions'
    WHEN product = 'Checking or savings account'
      AND issue = 'Problem with a lender or other company charging your account'
      AND sub_issue = 'Transaction was not authorized'
      THEN 'Disputes and unauthorized transactions'
    WHEN product = 'Checking or savings account' AND issue = 'Managing an account'
      AND sub_issue = 'Problem using a debit or ATM card'
      THEN 'Disputes and unauthorized transactions'
    WHEN product = 'Money transfer, virtual currency, or money service'
      AND issue = 'Unauthorized transactions or other transaction problem'
      THEN 'Disputes and unauthorized transactions'
    WHEN product = 'Money transfer, virtual currency, or money service' AND issue = 'Fraud or scam'
      THEN 'Fraud and scams'
    WHEN product = 'Checking or savings account' AND issue = 'Managing an account'
      AND sub_issue IN ('Deposits and withdrawals', 'Deposits or withdrawals', 'Problem accessing account')
      THEN 'Access to funds and holds'
    WHEN product = 'Money transfer, virtual currency, or money service'
      AND issue IN ('Trouble accessing funds in your mobile or digital wallet',
                    'Money was not available when promised',
                    'Managing, opening, or closing your mobile wallet account')
      THEN 'Access to funds and holds'
    WHEN product = 'Checking or savings account' AND issue = 'Managing an account'
      AND sub_issue IN ('Problem making or receiving payments', 'Funds not handled or disbursed as instructed')
      THEN 'Debits and payments gone wrong'
    WHEN product = 'Checking or savings account'
      AND issue = 'Problem with a lender or other company charging your account'
      AND (sub_issue LIKE 'Money was taken%' OR sub_issue LIKE '%stop withdrawals%')
      THEN 'Debits and payments gone wrong'
    WHEN product = 'Money transfer, virtual currency, or money service' AND issue = 'Other transaction problem'
      THEN 'Unclassified transaction problems'
    ELSE 'All other issues'
  END AS mode
FROM outcomes o;

-- 1. Failure modes: volume, two-year growth with and without Block and Zelle, share, outcomes
WITH b AS (
  SELECT mode,
    COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
    COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m,
    COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01'
                       AND company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')) AS last_xb,
    COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01'
                       AND company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')) AS prior_xb,
    ROUND(100.0 * AVG(got_relief::INT) FILTER (WHERE date_received >= DATE '2025-09-01'), 1) AS relief_pct,
    ROUND(100.0 * AVG(poor_outcome::INT) FILTER (WHERE date_received >= DATE '2025-09-01'), 1) AS poor_pct
  FROM modes GROUP BY mode
)
SELECT mode, last_11m, prior_11m,
       ROUND(100.0 * (last_11m - prior_11m) / NULLIF(prior_11m, 0), 1) AS growth_all,
       ROUND(100.0 * (last_xb - prior_xb) / NULLIF(prior_xb, 0), 1) AS growth_xb,
       ROUND(100.0 * last_11m / SUM(last_11m) OVER (), 1) AS share_pct,
       relief_pct, poor_pct
FROM b ORDER BY last_11m DESC;

-- 2. Who generates disputes and unauthorized complaints, and how each company resolves them
SELECT SUBSTR(m.company, 1, 28) AS company_short, COALESCE(cg.company_group, 'Other') AS grp,
       COUNT(*) AS complaints,
       ROUND(100.0 * AVG(m.got_relief::INT), 1) AS relief_pct,
       ROUND(100.0 * AVG(m.poor_outcome::INT), 1) AS poor_pct
FROM modes m LEFT JOIN company_groups cg ON m.company = cg.company
WHERE m.mode = 'Disputes and unauthorized transactions' AND m.date_received >= DATE '2025-09-01'
GROUP BY 1, 2
HAVING COUNT(*) >= 400
ORDER BY complaints DESC
LIMIT 15;

-- 3. Mode mix as a share of each group's complaints, last 11 months
WITH g AS (
  SELECT m.mode,
    CASE WHEN m.company LIKE 'Chime%' THEN 'Chime'
         WHEN cg.company_group = 'Bank' THEN 'Banks'
         WHEN cg.company_group = 'Fintech' AND m.company <> 'Block, Inc.' THEN 'Other fintech'
    END AS grp
  FROM modes m LEFT JOIN company_groups cg ON m.company = cg.company
  WHERE m.date_received >= DATE '2025-09-01'
),
c AS (
  SELECT mode,
    COUNT(*) FILTER (WHERE grp = 'Chime') AS chime,
    COUNT(*) FILTER (WHERE grp = 'Banks') AS banks,
    COUNT(*) FILTER (WHERE grp = 'Other fintech') AS other_fintech
  FROM g GROUP BY mode
)
SELECT mode, chime, ROUND(100.0 * chime / SUM(chime) OVER (), 1) AS chime_pct,
       banks, ROUND(100.0 * banks / SUM(banks) OVER (), 1) AS banks_pct,
       other_fintech, ROUND(100.0 * other_fintech / SUM(other_fintech) OVER (), 1) AS other_fintech_pct
FROM c ORDER BY chime DESC;

-- 4. Money transfer by sub-product without Block and Zelle
SELECT SUBSTR(issue, 1, 36) AS issue_short, SUBSTR(sub_product, 1, 30) AS sub_product_short,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
       COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m
FROM modes
WHERE product = 'Money transfer, virtual currency, or money service'
  AND company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')
  AND mode IN ('Fraud and scams', 'Unclassified transaction problems', 'Disputes and unauthorized transactions')
GROUP BY 1, 2
HAVING COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') >= 300
ORDER BY issue_short, last_11m DESC;
"""
Path('../sql/12_failure_modes.sql').write_text(modes_sql)

def run_text(name):
    text = Path(f'../sql/{name}').read_text()
    for stmt in text.split(';'):
        lines = stmt.splitlines()
        body = '\n'.join(l for l in lines if not l.strip().startswith('--')).strip()
        if not body:
            continue
        if body.upper().startswith(('SELECT', 'WITH')):
            labels = [l for l in lines if l.strip().startswith('--')]
            print('\n' + (labels[-1] if labels else ''))
            print(con.sql(stmt).df().to_string(index=False))
        else:
            con.sql(stmt)

run_text('12_failure_modes.sql')


-- 1. Failure modes: volume, two-year growth with and without Block and Zelle, share, outcomes
                                  mode  last_11m  prior_11m  growth_all  growth_xb  share_pct  relief_pct  poor_pct
                      All other issues     78751      53246        47.9       44.1       41.4        22.4      77.7
Disputes and unauthorized transactions     56175      26081       115.4      106.1       29.6        23.3      76.9
             Access to funds and holds     27430      14714        86.4       72.7       14.4        17.3      82.7
        Debits and payments gone wrong     10586       5153       105.4       99.2        5.6        14.5      85.5
                       Fraud and scams     10320       5201        98.4       66.5        5.4        10.3      89.8
     Unclassified transaction problems      6798       2470       175.2      106.6        3.6        10.3      89.9

-- 2. Who generates disputes and unauthorized complaints, and how each company resolves the

In [24]:
text = Path('../sql/12_failure_modes.sql').read_text()
for s in text.split(';'):
    if '-- 3.' in s or '-- 4.' in s:
        print(s.strip().splitlines()[0])
        print(con.sql(s).df().to_string(index=False))
        print()

print('-- disputes and unauthorized transactions by product')
print(con.sql("""
SELECT CASE product WHEN 'Checking or savings account' THEN 'Checking'
                    WHEN 'Money transfer, virtual currency, or money service' THEN 'Money transfer'
                    ELSE product END AS prod,
       COUNT(*) FILTER (WHERE date_received >= DATE '2025-09-01') AS last_11m,
       COUNT(*) FILTER (WHERE date_received < DATE '2024-08-01') AS prior_11m
FROM modes
WHERE mode = 'Disputes and unauthorized transactions'
GROUP BY 1 ORDER BY 2 DESC
""").df().to_string(index=False))

-- 3. Mode mix as a share of each group's complaints, last 11 months
                                  mode  chime  chime_pct  banks  banks_pct  other_fintech  other_fintech_pct
Disputes and unauthorized transactions   4406       41.9  22475       31.4           4498               27.1
                      All other issues   2302       21.9  32589       45.6           2854               17.2
             Access to funds and holds   1207       11.5   8365       11.7           5084               30.6
        Debits and payments gone wrong   1119       10.6   4360        6.1            607                3.7
                       Fraud and scams    800        7.6   2552        3.6           2094               12.6
     Unclassified transaction problems    690        6.6   1172        1.6           1477                8.9

-- 4. Money transfer by sub-product without Block and Zelle
                         issue_short            sub_product_short  last_11m  prior_11m
                    

In [25]:
print('-- same failure mode, same period: relief rate by company group (last 11 months)')
print(con.sql("""
WITH g AS (
  SELECT mode, got_relief, poor_outcome,
    CASE WHEN company LIKE 'Chime%' THEN 'chime'
         WHEN company LIKE 'Paypal%' THEN 'paypal'
         WHEN company = 'Block, Inc.' THEN 'block'
         WHEN company IN (SELECT company FROM company_groups WHERE company_group = 'Bank') THEN 'banks'
    END AS grp
  FROM modes WHERE date_received >= DATE '2025-09-01'
)
SELECT SUBSTR(mode, 1, 28) AS mode_short,
  COUNT(*) FILTER (WHERE grp = 'chime') AS chime_n,
  ROUND(100.0 * AVG(got_relief::INT) FILTER (WHERE grp = 'chime'), 1) AS chime_rel,
  COUNT(*) FILTER (WHERE grp = 'paypal') AS paypal_n,
  ROUND(100.0 * AVG(got_relief::INT) FILTER (WHERE grp = 'paypal'), 1) AS paypal_rel,
  COUNT(*) FILTER (WHERE grp = 'banks') AS banks_n,
  ROUND(100.0 * AVG(got_relief::INT) FILTER (WHERE grp = 'banks'), 1) AS banks_rel,
  COUNT(*) FILTER (WHERE grp = 'block') AS block_n,
  ROUND(100.0 * AVG(got_relief::INT) FILTER (WHERE grp = 'block'), 1) AS block_rel
FROM g WHERE grp IS NOT NULL
GROUP BY mode ORDER BY chime_n DESC
""").df().to_string(index=False))

-- same failure mode, same period: relief rate by company group (last 11 months)
                  mode_short  chime_n  chime_rel  paypal_n  paypal_rel  banks_n  banks_rel  block_n  block_rel
Disputes and unauthorized tr     4406        7.8      3081        45.9    22475       29.8     4256        0.1
            All other issues     2302        9.8      1191        34.8    32589       27.2     2333        0.1
   Access to funds and holds     1207        9.3      3605        33.3     8365       20.9     3026        0.0
Debits and payments gone wro     1119        7.8       234        35.0     4360       16.8      590        0.0
             Fraud and scams      800        3.3      1471        39.9     2552        8.9     2501        0.0
Unclassified transaction pro      690        1.4       737        37.4     1172       13.0     1099        0.0


In [26]:
from pathlib import Path

bench_sql = """-- 13_benchmark.sql
-- Purpose: compare relief rates across companies on the same failure-mode mix
-- Method: direct standardization. Each group's relief rate in each failure mode is weighted by
-- the banks' mode mix, so differences in what customers complain about drop out
-- Window: Sep 2025 to Jul 2026
-- Companies label their own responses, so this is an observation, not proof of service quality

CREATE OR REPLACE TABLE benchmark_by_mode AS
WITH g AS (
  SELECT mode, got_relief,
    CASE WHEN company LIKE 'Chime%' THEN 'Chime'
         WHEN company LIKE 'Paypal%' THEN 'PayPal'
         WHEN company = 'Block, Inc.' THEN 'Block'
         WHEN company IN (SELECT company FROM company_groups WHERE company_group = 'Bank') THEN 'Banks'
    END AS grp
  FROM modes WHERE date_received >= DATE '2025-09-01'
)
SELECT grp, mode, COUNT(*) AS n, AVG(got_relief::INT) AS relief
FROM g WHERE grp IS NOT NULL GROUP BY grp, mode;

CREATE OR REPLACE TABLE benchmark_rates AS
WITH weights AS (
  SELECT mode, n * 1.0 / SUM(n) OVER () AS w FROM benchmark_by_mode WHERE grp = 'Banks'
)
SELECT r.grp, SUM(r.n) AS complaints,
       ROUND(100.0 * SUM(r.n * r.relief) / SUM(r.n), 1) AS raw_relief_pct,
       ROUND(100.0 * SUM(w.w * r.relief), 1) AS relief_at_bank_mix_pct
FROM benchmark_by_mode r JOIN weights w USING (mode)
GROUP BY r.grp;

-- Result: raw and mix-adjusted relief rate by group
SELECT * FROM benchmark_rates ORDER BY relief_at_bank_mix_pct DESC;
"""
Path('../sql/13_benchmark.sql').write_text(bench_sql)

print('--- mix-adjusted benchmark (relief rate if each group had the banks complaint mix)')
for s in bench_sql.split(';'):
    body = '\n'.join(l for l in s.splitlines() if not l.strip().startswith('--')).strip()
    if not body:
        continue
    if body.upper().startswith('SELECT'):
        print(con.sql(s).df().to_string(index=False))
    else:
        con.sql(s)

out = Path('../data/processed')
out.mkdir(parents=True, exist_ok=True)

def save(name, df):
    df.to_csv(out / name, index=False)
    print(f'{name}: {len(df)} rows')

print('--- exports for Tableau')
save('monthly_trend.csv', con.sql("""
SELECT strftime(date_trunc('month', date_received), '%Y-%m-01') AS month, product,
       COUNT(*) AS complaints,
       COUNT(*) FILTER (WHERE NOT is_surge_event) AS complaints_ex_surge
FROM analysis GROUP BY 1, 2 ORDER BY 1, 2""").df())

save('mode_monthly.csv', con.sql("""
SELECT strftime(date_trunc('month', date_received), '%Y-%m-01') AS month, mode, COUNT(*) AS complaints
FROM modes GROUP BY 1, 2 ORDER BY 1, 2""").df())

modes_text = Path('../sql/12_failure_modes.sql').read_text()
for tag, name in [('-- 1.', 'failure_modes.csv'),
                  ('-- 2.', 'dispute_mode_by_company.csv'),
                  ('-- 3.', 'mode_mix_by_group.csv')]:
    stmt = [s for s in modes_text.split(';') if tag in s][0]
    save(name, con.sql(stmt).df())

save('benchmark_by_mode.csv', con.sql("""
SELECT grp, mode, n, ROUND(100.0 * relief, 1) AS relief_pct
FROM benchmark_by_mode ORDER BY grp, mode""").df())
save('benchmark_rates.csv', con.sql("SELECT * FROM benchmark_rates").df())

--- mix-adjusted benchmark (relief rate if each group had the banks complaint mix)
   grp  complaints  raw_relief_pct  relief_at_bank_mix_pct
PayPal     10319.0            38.5                    38.4
 Banks     71513.0            25.7                    25.7
 Chime     10524.0             7.6                     8.6
 Block     13805.0             0.1                     0.1
--- exports for Tableau
monthly_trend.csv: 140 rows
mode_monthly.csv: 210 rows
failure_modes.csv: 6 rows
dispute_mode_by_company.csv: 15 rows
mode_mix_by_group.csv: 6 rows
benchmark_by_mode.csv: 24 rows
benchmark_rates.csv: 4 rows


In [27]:
p = Path('../sql/06_growth.sql')
t = p.read_text()
note = "-- NOTE: superseded. This file compares one-year windows that the Jan 2025 surge distorts.\n-- See 09_priority.sql and 12_failure_modes.sql for the two-year method used in the findings.\n\n"
if 'superseded' not in t:
    p.write_text(note + t)
print('ok')

ok


In [28]:
from pathlib import Path

extract_py = '''import sys
from pathlib import Path
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
ARCHIVE = ROOT / "data" / "raw" / "archive"
OUT = ROOT / "data" / "processed" / "narratives_in_scope.csv"

PRODUCTS = [
    "Credit card",
    "Checking or savings account",
    "Money transfer, virtual currency, or money service",
    "Prepaid card",
]
START, END = "2023-09-01", "2026-08-14"
KEEP = ["Complaint ID", "Date received", "Product", "Sub-product", "Issue",
        "Sub-issue", "Company", "Consumer complaint narrative"]

OUT.parent.mkdir(parents=True, exist_ok=True)
files = sorted(ARCHIVE.glob("*.csv"))
print(f"{len(files)} files found", flush=True)

first, total, dupes = True, 0, 0
seen = set()
for fp in files:
    rows = kept = 0
    for chunk in pd.read_csv(fp, chunksize=200_000, dtype=str):
        rows += len(chunk)
        dates = pd.to_datetime(chunk["Date received"], errors="coerce")
        narr = chunk["Consumer complaint narrative"].fillna("").str.strip()
        mask = (chunk["Product"].isin(PRODUCTS) & (dates >= START)
                & (dates <= END) & (narr != ""))
        sub = chunk.loc[mask, KEEP]
        fresh = ~sub["Complaint ID"].isin(seen)
        dupes += int((~fresh).sum())
        sub = sub[fresh]
        seen.update(sub["Complaint ID"])
        sub.to_csv(OUT, mode="w" if first else "a", header=first, index=False)
        first = False
        kept += len(sub)
    total += kept
    print(f"{fp.name}: {rows:,} rows, {kept:,} kept, running total {total:,}", flush=True)

print(f"Done. {total:,} narratives saved to {OUT} ({dupes} duplicates dropped)")
'''
Path('../scripts/extract_narratives.py').write_text(extract_py)
print('written')

written


In [29]:
con.sql("""
CREATE OR REPLACE TABLE narratives AS
SELECT CAST("Complaint ID" AS BIGINT) AS complaint_id,
       "Consumer complaint narrative" AS narrative
FROM read_csv_auto('../data/processed/narratives_in_scope.csv', header=true, all_varchar=true)
""")

print('--- narratives loaded, and how many match your scoped complaints')
print(con.sql("""
SELECT (SELECT COUNT(*) FROM narratives) AS narratives_loaded,
       COUNT(*) AS matched_to_scoped
FROM narratives n JOIN complaints_scoped c USING (complaint_id)
""").df().to_string(index=False))

print('--- narrative coverage, percent of in-scope complaints that have one (rows are months, columns are years)')
print(con.sql("""
SELECT EXTRACT(month FROM c.date_received)::INT AS m,
  ROUND(100.0 * COUNT(n.complaint_id) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2023)
        / NULLIF(COUNT(*) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2023), 0), 1) AS y2023,
  ROUND(100.0 * COUNT(n.complaint_id) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2024)
        / NULLIF(COUNT(*) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2024), 0), 1) AS y2024,
  ROUND(100.0 * COUNT(n.complaint_id) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2025)
        / NULLIF(COUNT(*) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2025), 0), 1) AS y2025,
  ROUND(100.0 * COUNT(n.complaint_id) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2026)
        / NULLIF(COUNT(*) FILTER (WHERE EXTRACT(year FROM c.date_received) = 2026), 0), 1) AS y2026
FROM complaints_scoped c LEFT JOIN narratives n USING (complaint_id)
GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

print('--- narratives by Block and Zelle in the campaign months')
print(con.sql("""
SELECT strftime(date_trunc('month', c.date_received), '%Y-%m') AS month,
       SUBSTR(c.company, 1, 12) AS company_short,
       COUNT(*) AS in_scope, COUNT(n.complaint_id) AS with_narrative
FROM complaints_scoped c LEFT JOIN narratives n USING (complaint_id)
WHERE c.company IN ('Block, Inc.', 'Early Warning Services, LLC')
  AND c.date_received BETWEEN DATE '2024-12-01' AND DATE '2025-04-30'
GROUP BY 1, 2 ORDER BY 2, 1
""").df().to_string(index=False))

--- narratives loaded, and how many match your scoped complaints
 narratives_loaded  matched_to_scoped
            318648             297905
--- narrative coverage, percent of in-scope complaints that have one (rows are months, columns are years)
 m  y2023  y2024  y2025  y2026
 1    NaN   56.1   79.3   47.6
 2    NaN   56.4   65.8   30.4
 3    NaN   55.8   60.7   44.6
 4    NaN   55.5   60.9   49.0
 5    NaN   57.3   62.3   42.1
 6    NaN   58.2   62.1   41.6
 7    NaN   57.4   62.8   24.0
 8    NaN   56.5   61.9    0.0
 9   58.8   56.2   58.6    NaN
10   57.0   57.0   56.0    NaN
11   57.4   57.7   60.1    NaN
12   57.9   59.3   43.7    NaN
--- narratives by Block and Zelle in the campaign months
  month company_short  in_scope  with_narrative
2024-12   Block, Inc.       358             246
2025-01   Block, Inc.     31221           26302
2025-02   Block, Inc.      3822            2966
2025-03   Block, Inc.      1710            1257
2025-04   Block, Inc.      1295             942
2024-

In [30]:
from pathlib import Path

test_py = '''import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Purpose: measure how templated complaint narratives are, by group
# Method: normalize text, then measure exact duplicates, shared 10-word openings,
# and the share of narratives whose nearest neighbor within the same group has cosine similarity >= 0.8
# Each group is sampled to the same size (3000), so rates are comparable across groups
# Narratives under 20 words are dropped because very short text cannot be templated in a meaningful way

def norm(t):
    t = re.sub(r"\\bx{2,}\\b", " ", t.lower())
    t = re.sub(r"[^a-z0-9 ]+", " ", t)
    return re.sub(r"\\s+", " ", t).strip()

def campaign_test(con, sample_size=3000, seed=1):
    df = con.sql("""
    SELECT * FROM (
      SELECT c.complaint_id, c.date_received, c.company, n.narrative,
        CASE
          WHEN c.company IN ('Block, Inc.', 'Early Warning Services, LLC')
               AND c.date_received BETWEEN DATE '2023-09-01' AND DATE '2024-12-31'
            THEN '0 Block+Zelle before (Sep 2023-Dec 2024)'
          WHEN c.company IN ('Block, Inc.', 'Early Warning Services, LLC')
               AND c.date_received BETWEEN DATE '2025-01-13' AND DATE '2025-02-28'
            THEN '1 Block+Zelle campaign (Jan 13-Feb 28 2025)'
          WHEN c.company IN ('Block, Inc.', 'Early Warning Services, LLC')
               AND c.date_received BETWEEN DATE '2025-03-01' AND DATE '2025-08-31'
            THEN '2 Block+Zelle Mar-Aug 2025'
          WHEN c.company IN ('Block, Inc.', 'Early Warning Services, LLC')
               AND c.date_received >= DATE '2025-09-01'
            THEN '3 Block+Zelle after (Sep 2025-Jul 2026)'
          WHEN c.company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')
               AND c.company NOT IN (SELECT company FROM spike_companies)
               AND c.date_received BETWEEN DATE '2025-01-13' AND DATE '2025-02-28'
            THEN '4 Control: all others, same weeks'
          WHEN c.company NOT IN ('Block, Inc.', 'Early Warning Services, LLC')
               AND c.date_received >= DATE '2025-09-01'
            THEN '5 Control: all others, Sep 2025-Jul 2026'
        END AS grp
      FROM narratives n JOIN complaints_scoped c USING (complaint_id)
    ) WHERE grp IS NOT NULL
    """).df()
    df = df[df["narrative"].str.split().str.len() >= 20]
    rows, openers = [], {}
    for g, d in sorted(df.groupby("grp"), key=lambda x: x[0]):
        d = d.sample(min(len(d), sample_size), random_state=seed)
        texts = d["narrative"].map(norm).tolist()
        words = [t.split() for t in texts]
        exact = pd.Series(texts).duplicated(keep=False).mean()
        op = pd.Series([" ".join(w[:10]) for w in words])
        X = TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True).fit_transform(texts)
        nn = NearestNeighbors(n_neighbors=2, metric="cosine", algorithm="brute").fit(X)
        dist, _ = nn.kneighbors(X)
        sim = 1 - dist[:, 1]
        rows.append(dict(group=g, narratives=len(texts),
                         median_words=int(np.median([len(w) for w in words])),
                         exact_dup_pct=round(100 * exact, 1),
                         same_opening_pct=round(100 * op.duplicated(keep=False).mean(), 1),
                         near_dup_pct=round(100 * float((sim >= 0.8).mean()), 1)))
        openers[g] = op.value_counts().head(3)
    return pd.DataFrame(rows), openers
'''
Path('../scripts/campaign_test.py').write_text(test_py)

import sys, importlib
sys.path.insert(0, '../scripts')
import campaign_test
importlib.reload(campaign_test)

res, openers = campaign_test.campaign_test(con)
print('--- how templated are the narratives? (3000-narrative samples, same size in every group)')
print(res.to_string(index=False))
print()
for g in res['group']:
    if g.startswith(('1', '3')):
        print('most common 10-word openings,', g)
        print(openers[g].to_string())
        print()

print('--- narratives available by failure mode (surge rows and in-progress complaints excluded)')
print(con.sql("""
SELECT m.mode, COUNT(*) AS resolved_in_scope, COUNT(n.complaint_id) AS with_narrative,
       ROUND(100.0 * COUNT(n.complaint_id) / COUNT(*), 1) AS pct
FROM modes m LEFT JOIN narratives n USING (complaint_id)
GROUP BY 1 ORDER BY 2 DESC
""").df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- how templated are the narratives? (3000-narrative samples, same size in every group)
                                      group  narratives  median_words  exact_dup_pct  same_opening_pct  near_dup_pct
   0 Block+Zelle before (Sep 2023-Dec 2024)        3000           136            0.5               2.8           3.0
1 Block+Zelle campaign (Jan 13-Feb 28 2025)        3000            95           55.5              84.5          78.0
                 2 Block+Zelle Mar-Aug 2025        3000            95           15.7              27.1          26.3
    3 Block+Zelle after (Sep 2025-Jul 2026)        3000           124            4.1              10.7           9.0
          4 Control: all others, same weeks        3000           149            1.1               3.3           3.5
   5 Control: all others, Sep 2025-Jul 2026        3000           186            0.1               2.0           1.0

most common 10-word openings, 1 Block+Zelle campaign (Jan 13-Feb 28 2025)
i am filing a com

In [31]:
print(con.sql("""
SELECT CASE WHEN company = 'Block, Inc.' THEN 'Block' ELSE 'Zelle (EWS)' END AS co,
       CASE WHEN date_received < DATE '2025-01-13' THEN '1 before campaign'
            WHEN date_received <= DATE '2025-08-31' THEN '2 campaign window'
            ELSE '3 after (Sep 2025 on)' END AS period,
       COUNT(*) AS complaints,
       ROUND(100.0 * AVG((company_response IN ('Closed with monetary relief',
                          'Closed with non-monetary relief'))::INT), 1) AS relief_pct
FROM complaints_scoped
WHERE company IN ('Block, Inc.', 'Early Warning Services, LLC')
  AND company_response <> 'In progress'
GROUP BY 1, 2 ORDER BY 1, 2
""").df().to_string(index=False))

         co                period  complaints  relief_pct
      Block     1 before campaign        4469         0.6
      Block     2 campaign window       42862         0.0
      Block 3 after (Sep 2025 on)       14299         0.1
Zelle (EWS)     1 before campaign        1104         0.0
Zelle (EWS)     2 campaign window       19074         0.0
Zelle (EWS) 3 after (Sep 2025 on)        2258         0.0


In [32]:
from pathlib import Path

themes_py = r'''import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

# Purpose: find the recurring themes inside complaint narratives for each failure mode
# Window: complaints received Sep 2023 to Nov 2025, the months where narrative coverage is stable
# Cleaning, applied before modeling:
#   1. surge-flagged rows are already out (the modes view excludes them)
#   2. drop narratives under 30 words
#   3. drop Block and Early Warning Services narratives that start with one of the two template openings
#      found by the campaign test
#   4. drop exact duplicates after normalizing the text
# Model: TF-IDF (single words and word pairs) then NMF, which gives each narrative one dominant theme
# Fixed random seed. Each mode is capped at 20,000 randomly sampled narratives
# Themes are labeled by hand after reading the top terms and 20 sample narratives per theme

K = {
    "Disputes and unauthorized transactions": 8,
    "Access to funds and holds": 6,
    "Debits and payments gone wrong": 5,
    "Unclassified transaction problems": 6,
}
MAX_N = 20000
BLOCK_ZELLE = ("Block, Inc.", "Early Warning Services, LLC")
TEMPLATE_STARTS = ("i am filing a complaint against cash app block inc",
                   "i am writing to formally express my concerns regarding")


def norm(t):
    t = re.sub(r"\bx{2,}\b", " ", t.lower())
    t = re.sub(r"[^a-z ]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def run_themes(con, out_dir="../data/processed"):
    df = con.sql("""
    SELECT m.complaint_id, m.mode, m.company, m.date_received, m.got_relief, n.narrative
    FROM modes m JOIN narratives n USING (complaint_id)
    WHERE m.date_received BETWEEN DATE '2023-09-01' AND DATE '2025-11-30'
      AND m.mode IN ('Disputes and unauthorized transactions', 'Access to funds and holds',
                     'Debits and payments gone wrong', 'Unclassified transaction problems')
    """).df()
    rows, samples = [], []
    for mode, k in K.items():
        d = df[df["mode"] == mode].copy()
        start_n = len(d)
        d["clean"] = d["narrative"].map(norm)
        d = d[d["clean"].str.split().str.len() >= 30]
        short_n = start_n - len(d)
        is_tmpl = d["company"].isin(BLOCK_ZELLE) & d["clean"].str.startswith(TEMPLATE_STARTS)
        tmpl_n = int(is_tmpl.sum())
        d = d[~is_tmpl]
        before = len(d)
        d = d.drop_duplicates("clean")
        dup_n = before - len(d)
        if len(d) > MAX_N:
            d = d.sample(MAX_N, random_state=1)
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5,
                              max_df=0.4, max_features=30000, sublinear_tf=True)
        X = vec.fit_transform(d["clean"])
        nmf = NMF(n_components=k, init="nndsvd", random_state=1, max_iter=300)
        d["theme"] = nmf.fit_transform(X).argmax(axis=1)
        terms = vec.get_feature_names_out()
        print(f"{mode}: {start_n:,} narratives, dropped {short_n:,} short, {tmpl_n:,} template, {dup_n:,} duplicates, modeled {len(d):,}", flush=True)
        for t in range(k):
            top = ", ".join(terms[nmf.components_[t].argsort()[::-1][:10]])
            sub = d[d["theme"] == t]
            rows.append(dict(mode=mode, theme=t, narratives=len(sub),
                             share_pct=round(100 * len(sub) / len(d), 1),
                             relief_pct=round(100 * float(sub["got_relief"].mean()), 1),
                             top_terms=top))
            samples.append(sub.sample(min(20, len(sub)), random_state=1)
                              [["mode", "theme", "complaint_id", "company", "narrative"]])
    summary = pd.DataFrame(rows)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    summary.to_csv(Path(out_dir) / "theme_summary.csv", index=False)
    pd.concat(samples).to_csv(Path(out_dir) / "theme_samples.csv", index=False)
    return summary


def show(summary, mode_prefixes):
    for _, r in summary.iterrows():
        if r["mode"].startswith(mode_prefixes):
            print(f'{r["mode"][:12]} T{r["theme"]}: {r["share_pct"]:4.1f}% of narratives | relief {r["relief_pct"]:4.1f}% | {r["top_terms"]}')
'''
Path('../scripts/themes.py').write_text(themes_py)

import sys, importlib
sys.path.insert(0, '../scripts')
import themes
importlib.reload(themes)
summary = themes.run_themes(con)
print()
themes.show(summary, ('Disputes', 'Access'))

Disputes and unauthorized transactions: 42,491 narratives, dropped 1,482 short, 3 template, 57 duplicates, modeled 20,000
Access to funds and holds: 25,130 narratives, dropped 971 short, 1 template, 42 duplicates, modeled 20,000
Debits and payments gone wrong: 9,552 narratives, dropped 429 short, 0 template, 8 duplicates, modeled 9,115


/Users/mansi/Documents/GitHub/cfpd-complaints-intelligence/venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 300 reached. Increase it to improve convergence.
  warnings.warn(


Unclassified transaction problems: 6,046 narratives, dropped 263 short, 483 template, 139 duplicates, modeled 5,161

Disputes and T0: 20.8% of narratives | relief 24.4% | money, told, said, called, got, help, just, days, went, asked
Disputes and T1: 14.3% of narratives | relief 22.7% | financial, consumer, despite, complaint, matter, resolution, unauthorized, cfpb, act, protection
Disputes and T2: 13.6% of narratives | relief 20.4% | transactions, police, claim, report, denied, debit, chime, debit card, police report, unauthorized
Disputes and T3:  6.2% of narratives | relief 25.3% | wells, fargo, wells fargo, claim, fargo bank, called wells, fargo denied, account wells, contacted wells, debit
Disputes and T4:  3.7% of narratives | relief  7.1% | cash app, cash, app, app account, money, cashapp, transaction, app card, contacted cash, funds
Disputes and T5: 17.0% of narratives | relief 38.8% | charges, credit card, charge, fraudulent, fraudulent charges, fraud, charged, balance, company

In [33]:
themes.show(summary, ('Debits', 'Unclass'))

Debits and p T0: 29.7% of narratives | relief 21.4% | check, told, chase, called, said, sent, deposited, branch, received, number
Debits and p T1:  8.2% of narratives | relief 15.4% | fargo, wells, wells fargo, fargo bank, claim, account wells, fargo account, called wells, checking, fraud
Debits and p T2: 21.5% of narratives | relief 16.7% | financial, complaint, despite, matter, regarding, provide, resolution, issue, cfpb, request
Debits and p T3: 33.6% of narratives | relief 17.9% | card, chime, dispute, credit, transaction, refund, debit, did, charge, claim
Debits and p T4:  7.0% of narratives | relief 26.8% | bonus, direct, deposit, direct deposit, opened, offer, checking, promotion, deposits, requirements
Unclassified T0:  4.7% of narratives | relief  0.8% | chime, concerns, using chime, chime neglected, chime numerous, chime failed, cfpb, formally express, consumers, indicate chime
Unclassified T1: 40.5% of narratives | relief 15.3% | told, said, sent, called, did, paypal, card, 

In [34]:
from pathlib import Path

themes_py = r'''import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

# Purpose: find the recurring themes inside complaint narratives for each failure mode
# Window: complaints received Sep 2023 to Nov 2025, the months where narrative coverage is stable
# Cleaning, applied before modeling:
#   1. surge-flagged rows are already out (the modes view excludes them)
#   2. drop narratives under 30 words
#   3. drop Block and Early Warning Services narratives that start with one of the two template openings
#      found by the campaign test
#   4. drop exact duplicates after normalizing the text
# Model: TF-IDF (single words and word pairs) then NMF, which gives each narrative one dominant theme
# Version 2 changes: company and bank names plus common narrative verbs are added to the stop words, so themes
# describe what went wrong and not who it happened to; NMF runs up to 1000 iterations;
# Fraud and scams is modeled too; each theme reports the share of its narratives that share an 8-word
# opening with at least 10 other narratives in the same mode (a simple template check)
# Fixed random seed. Each mode is capped at 20,000 randomly sampled narratives
# Themes are labeled by hand after reading the top terms and 20 sample narratives per theme

K = {
    "Disputes and unauthorized transactions": 8,
    "Access to funds and holds": 6,
    "Debits and payments gone wrong": 5,
    "Fraud and scams": 6,
    "Unclassified transaction problems": 6,
}
MAX_N = 20000
BLOCK_ZELLE = ("Block, Inc.", "Early Warning Services, LLC")
TEMPLATE_STARTS = ("i am filing a complaint against cash app block inc",
                   "i am writing to formally express my concerns regarding")
COMPANY_WORDS = {"wells", "fargo", "chase", "jpmorgan", "morgan", "bank", "banks", "america", "bofa", "boa",
                 "citi", "citibank", "capital", "chime", "cash", "app", "cashapp", "paypal", "venmo",
                 "coinbase", "zelle", "zelles", "block", "robinhood", "sofi", "wise", "remitly", "navy",
                 "federal", "usaa", "discover", "synchrony", "amex", "american", "express", "goldman",
                 "sachs", "apple", "dave", "moneylion"}
GENERIC_WORDS = {"told", "said", "called", "asked", "tried", "just", "got", "did", "went", "like", "know",
                 "say", "says", "didn", "doesn", "wasn", "don", "ve", "ll", "pm", "am"}
STOP = sorted(set(ENGLISH_STOP_WORDS) | COMPANY_WORDS | GENERIC_WORDS)


def norm(t):
    t = re.sub(r"\bx{2,}\b", " ", t.lower())
    t = re.sub(r"[^a-z ]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def run_themes(con, out_dir="../data/processed"):
    in_list = ", ".join("'" + m + "'" for m in K)
    df = con.sql(f"""
    SELECT m.complaint_id, m.mode, m.company, m.date_received, m.got_relief, n.narrative
    FROM modes m JOIN narratives n USING (complaint_id)
    WHERE m.date_received BETWEEN DATE '2023-09-01' AND DATE '2025-11-30'
      AND m.mode IN ({in_list})
    """).df()
    rows, samples = [], []
    for mode, k in K.items():
        d = df[df["mode"] == mode].copy()
        start_n = len(d)
        d["clean"] = d["narrative"].map(norm)
        d = d[d["clean"].str.split().str.len() >= 30]
        short_n = start_n - len(d)
        is_tmpl = d["company"].isin(BLOCK_ZELLE) & d["clean"].str.startswith(TEMPLATE_STARTS)
        tmpl_n = int(is_tmpl.sum())
        d = d[~is_tmpl]
        before = len(d)
        d = d.drop_duplicates("clean")
        dup_n = before - len(d)
        d["open8"] = d["clean"].str.split().str[:8].str.join(" ")
        d["open_n"] = d.groupby("open8")["open8"].transform("size")
        if len(d) > MAX_N:
            d = d.sample(MAX_N, random_state=1)
        vec = TfidfVectorizer(stop_words=STOP, ngram_range=(1, 2), min_df=5,
                              max_df=0.4, max_features=30000, sublinear_tf=True)
        X = vec.fit_transform(d["clean"])
        nmf = NMF(n_components=k, init="nndsvd", random_state=1, max_iter=1000)
        d["theme"] = nmf.fit_transform(X).argmax(axis=1)
        terms = vec.get_feature_names_out()
        print(f"{mode}: {start_n:,} narratives, dropped {short_n:,} short, {tmpl_n:,} template, "
              f"{dup_n:,} duplicates, modeled {len(d):,}", flush=True)
        for t in range(k):
            top = ", ".join(terms[nmf.components_[t].argsort()[::-1][:10]])
            sub = d[d["theme"] == t]
            rows.append(dict(mode=mode, theme=t, narratives=len(sub),
                             share_pct=round(100 * len(sub) / len(d), 1),
                             relief_pct=round(100 * float(sub["got_relief"].mean()), 1),
                             shared_opening_pct=round(100 * float((sub["open_n"] >= 10).mean()), 1),
                             top_terms=top))
            samples.append(sub.sample(min(20, len(sub)), random_state=1)
                              [["mode", "theme", "complaint_id", "company", "narrative"]])
    summary = pd.DataFrame(rows)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    summary.to_csv(Path(out_dir) / "theme_summary.csv", index=False)
    pd.concat(samples).to_csv(Path(out_dir) / "theme_samples.csv", index=False)
    return summary


def show(summary, mode_prefixes):
    for _, r in summary.iterrows():
        if r["mode"].startswith(mode_prefixes):
            print(f'{r["mode"][:12]} T{r["theme"]}: {r["share_pct"]:4.1f}% | relief {r["relief_pct"]:4.1f}% '
                  f'| shared opening {r["shared_opening_pct"]:4.1f}% | {r["top_terms"]}')
'''
Path('../scripts/themes.py').write_text(themes_py)

import sys, importlib
sys.path.insert(0, '../scripts')
import themes
importlib.reload(themes)
summary = themes.run_themes(con)
print()
themes.show(summary, ('Disputes', 'Access'))

Disputes and unauthorized transactions: 42,491 narratives, dropped 1,482 short, 3 template, 57 duplicates, modeled 20,000
Access to funds and holds: 25,130 narratives, dropped 971 short, 1 template, 42 duplicates, modeled 20,000
Debits and payments gone wrong: 9,552 narratives, dropped 429 short, 0 template, 8 duplicates, modeled 9,115
Fraud and scams: 9,059 narratives, dropped 471 short, 5 template, 27 duplicates, modeled 8,556
Unclassified transaction problems: 6,046 narratives, dropped 263 short, 483 template, 139 duplicates, modeled 5,161

Disputes and T0:  9.6% | relief 30.6% | shared opening  0.6% | email, received, number, phone, information, customer, letter, sent, customer service, service
Disputes and T1: 14.5% | relief 23.0% | shared opening  9.0% | despite, consumer, financial, complaint, matter, resolution, cfpb, documentation, evidence, regarding
Disputes and T2: 10.8% | relief 37.4% | shared opening  0.3% | charges, fraudulent, fraudulent charges, fraud, purchases, credi

In [35]:
themes.show(summary, ('Debits', 'Fraud', 'Unclass'))

Debits and p T0: 21.6% | relief 19.7% | shared opening  0.1% | card, credit, dispute, payment, refund, debit, transaction, charge, claim, charged
Debits and p T1: 16.8% | relief 16.0% | shared opening  3.5% | financial, complaint, despite, matter, regarding, provide, cfpb, resolution, consumer, request
Debits and p T2: 15.4% | relief 21.1% | shared opening  0.0% | check, deposited, cashed, deposit, checks, received, closed, hold, payment, sent
Debits and p T3:  6.8% | relief 27.1% | shared opening  0.5% | bonus, direct, deposit, direct deposit, opened, checking, offer, deposits, promotion, checking account
Debits and p T4: 39.4% | relief 17.9% | shared opening  0.1% | phone, number, transfer, fraud, help, sent, time, email, information, access
Fraud and sc T0: 42.6% | relief 11.2% | shared opening  0.1% | phone, number, card, send, email, person, help, received, time, information
Fraud and sc T1:  3.6% | relief  2.3% | shared opening 72.8% | involved making, deposits total, making depo

In [36]:
base = """
WITH g AS (
  SELECT mode, product, got_relief,
    CASE WHEN company LIKE 'Chime%' THEN 'Chime'
         WHEN company IN (SELECT company FROM company_groups WHERE company_group = 'Bank') THEN 'Banks'
    END AS grp
  FROM modes WHERE date_received >= DATE '2025-09-01'
),
cells AS (
  SELECT product, mode,
    COUNT(*) FILTER (WHERE grp = 'Chime') AS chime_n,
    AVG(got_relief::INT) FILTER (WHERE grp = 'Chime') AS chime_rel,
    COUNT(*) FILTER (WHERE grp = 'Banks') AS banks_n,
    AVG(got_relief::INT) FILTER (WHERE grp = 'Banks') AS banks_rel
  FROM g WHERE grp IS NOT NULL GROUP BY product, mode
)
"""
print('--- same product, same failure mode: Chime vs banks (cells where Chime has 100+ complaints)')
print(con.sql(base + """
SELECT CASE product WHEN 'Checking or savings account' THEN 'Checking'
                    WHEN 'Money transfer, virtual currency, or money service' THEN 'Money transfer'
                    ELSE product END AS prod,
       SUBSTR(mode, 1, 30) AS mode_short, chime_n,
       ROUND(100.0 * chime_rel, 1) AS chime_rel, banks_n, ROUND(100.0 * banks_rel, 1) AS banks_rel
FROM cells WHERE chime_n >= 100 ORDER BY chime_n DESC
""").df().to_string(index=False))
print()
print('--- Chime vs banks re-weighted to Chime product and mode mix (cells with 30+ complaints on both sides)')
print(con.sql(base + """
SELECT SUM(chime_n) AS chime_complaints,
       ROUND(100.0 * SUM(chime_n * chime_rel) / SUM(chime_n), 1) AS chime_relief_pct,
       ROUND(100.0 * SUM(chime_n * banks_rel) / SUM(chime_n), 1) AS banks_at_chime_mix_pct
FROM cells WHERE chime_n >= 30 AND banks_n >= 30
""").df().to_string(index=False))

--- same product, same failure mode: Chime vs banks (cells where Chime has 100+ complaints)
          prod                     mode_short  chime_n  chime_rel  banks_n  banks_rel
      Checking Disputes and unauthorized tran     3298        7.8     8760       23.5
      Checking               All other issues     1795       10.7    11401       24.3
      Checking Debits and payments gone wrong     1119        7.8     4360       16.8
      Checking      Access to funds and holds     1078        9.1     7907       21.2
Money transfer                Fraud and scams      800        3.3     2552        8.9
Money transfer Unclassified transaction probl      690        1.4     1172       13.0
   Credit card Disputes and unauthorized tran      581        9.5    13279       34.5
Money transfer Disputes and unauthorized tran      419        4.8      309       12.6
   Credit card               All other issues      287        7.0    20392       29.4
Money transfer               All other issues   

In [37]:
from pathlib import Path

prod_sql = """-- 14_benchmark_product.sql
-- Purpose: compare Chime with banks inside the same product and failure mode, then re-weight banks to Chime's mix
-- This replaces the mode-only comparison in 13_benchmark.sql, which does not adjust for product
-- Window: Sep 2025 to Jul 2026. Companies label their own responses, so gaps are observations, not proof of service quality

-- 1. Cells: relief rate by product and failure mode, Chime vs banks
CREATE OR REPLACE TABLE chime_vs_banks AS
WITH g AS (
  SELECT mode, product, got_relief,
    CASE WHEN company LIKE 'Chime%' THEN 'Chime'
         WHEN company IN (SELECT company FROM company_groups WHERE company_group = 'Bank') THEN 'Banks'
    END AS grp
  FROM modes WHERE date_received >= DATE '2025-09-01'
)
SELECT product, mode,
  COUNT(*) FILTER (WHERE grp = 'Chime') AS chime_n,
  AVG(got_relief::INT) FILTER (WHERE grp = 'Chime') AS chime_rel,
  COUNT(*) FILTER (WHERE grp = 'Banks') AS banks_n,
  AVG(got_relief::INT) FILTER (WHERE grp = 'Banks') AS banks_rel
FROM g WHERE grp IS NOT NULL GROUP BY product, mode;

-- 2. Banks re-weighted to Chime's product and mode mix
SELECT SUM(chime_n) AS chime_complaints,
       ROUND(100.0 * SUM(chime_n * chime_rel) / SUM(chime_n), 1) AS chime_relief_pct,
       ROUND(100.0 * SUM(chime_n * banks_rel) / SUM(chime_n), 1) AS banks_at_chime_mix_pct
FROM chime_vs_banks WHERE chime_n >= 30 AND banks_n >= 30;
"""
Path('../sql/14_benchmark_product.sql').write_text(prod_sql)

for s in prod_sql.split(';'):
    body = '\n'.join(l for l in s.splitlines() if not l.strip().startswith('--')).strip()
    if not body:
        continue
    if body.upper().startswith('SELECT'):
        print(con.sql(s).df().to_string(index=False))
    else:
        con.sql(s)

con.sql("""SELECT product, mode, chime_n, ROUND(100.0 * chime_rel, 1) AS chime_relief_pct,
                  banks_n, ROUND(100.0 * banks_rel, 1) AS banks_relief_pct
           FROM chime_vs_banks ORDER BY chime_n DESC""").df().to_csv('../data/processed/chime_vs_banks_cells.csv', index=False)
print('saved')

 chime_complaints  chime_relief_pct  banks_at_chime_mix_pct
          10524.0               7.6                    20.7
saved


In [38]:
print(con.sql("SELECT COUNT(*) AS narratives FROM narratives").df().to_string(index=False))

 narratives
     318648


In [39]:
from pathlib import Path

validate_py = r'''import pandas as pd
from pathlib import Path

# Purpose: manual check of theme labels
# For each theme below, show 10 sampled narratives (first 70 words) and record whether the main complaint
# fits the proposed label (1) or not (0)
# Rule: 1 if the main complaint is what the label says, even if the narrative also mentions other things
# 0 if the main complaint is something else or the narrative is unclear
# Answers are saved after every entry, so the check can be stopped (q) and resumed
# Type f to see the full narrative before answering

CHECK = [
    ("Disputes and unauthorized transactions", 5, "Unauthorized debit card transactions, claim denied"),
    ("Disputes and unauthorized transactions", 6, "Money taken from a checking account or ATM, no help"),
    ("Disputes and unauthorized transactions", 7, "Merchant refund or non-delivery dispute"),
    ("Disputes and unauthorized transactions", 3, "Charge disputed with a company after payment"),
    ("Access to funds and holds", 2, "Check deposit held, funds not available"),
    ("Access to funds and holds", 0, "Locked out of account or stuck in verification"),
    ("Access to funds and holds", 4, "Account closed or funds hard to withdraw (tentative, catch-all cluster)"),
    ("Debits and payments gone wrong", 3, "Sign-up bonus or direct deposit promotion not honored"),
    ("Fraud and scams", 5, "Fake check or job scam"),
    ("Unclassified transaction problems", 3, "Wire transfer problem, recall requested"),
]
SAMPLES = Path("../data/processed/theme_samples.csv")
ANSWERS = Path("../data/processed/validation_answers.csv")


def run(per_theme=10, words=70):
    samples = pd.read_csv(SAMPLES)
    if ANSWERS.exists():
        done = pd.read_csv(ANSWERS)
    else:
        done = pd.DataFrame(columns=["mode", "theme", "complaint_id", "fits"])
    answered = set(zip(done["mode"], done["theme"], done["complaint_id"]))
    for mode, theme, label in CHECK:
        s = samples[(samples["mode"] == mode) & (samples["theme"] == theme)].head(per_theme)
        for _, r in s.iterrows():
            key = (mode, theme, r["complaint_id"])
            if key in answered:
                continue
            text = " ".join(str(r["narrative"]).split()[:words])
            print(f"\n[{mode[:12]} T{theme}] LABEL: {label}\n{text}")
            ans = input("Fits the label? 1 yes, 0 no, f full text, q stop: ").strip().lower()
            if ans == "f":
                print("\nFULL TEXT:\n" + str(r["narrative"]))
                ans = input("Fits the label? 1 yes, 0 no, q stop: ").strip().lower()
            if ans == "q":
                print("Stopped. Run again to resume.")
                return
            if ans not in ("0", "1"):
                print("Skipped, run again to answer it later.")
                continue
            done.loc[len(done)] = [mode, theme, r["complaint_id"], int(ans)]
            done.to_csv(ANSWERS, index=False)
    print("All answered.")


def summary():
    d = pd.read_csv(ANSWERS)
    labels = {(m, t): l for m, t, l in CHECK}
    g = d.groupby(["mode", "theme"])["fits"].agg(["sum", "count"]).reset_index()
    for _, r in g.iterrows():
        name = labels[(r["mode"], r["theme"])]
        print(f"{r['mode'][:12]} T{r['theme']}: {int(r['sum'])} of {int(r['count'])} fit | {name}")
'''
Path('../scripts/validate.py').write_text(validate_py)

import sys, importlib
sys.path.insert(0, '../scripts')
import validate
importlib.reload(validate)
validate.run()


[Disputes and T5] LABEL: Unauthorized debit card transactions, claim denied
XXXX XXXX ( XXXX XXXX ) and XXXX XXXX ( all information Ive found about this merchant is that its a scam ) allowed fraudulent transactions to be posted on their card batches. I filed a fraud claim with my bank, Fifth Third, for those two transactions and a third transaction that was already stopped by the merchant for being fraudulent and is no longer part of the fraud investigation

[Disputes and T5] LABEL: Unauthorized debit card transactions, claim denied
So I filed a claim for fraud a transaction merchant name XXXX that was unauthorized and come to find out that they ended up closing my account saying I violated the the terms of service after I told them that there was a transaction that was made with them with my card that I didnt authorize the amount of the transaction is {$160.00}

[Disputes and T5] LABEL: Unauthorized debit card transactions, claim denied
On XX/XX/scrub> 2025, I discovered {$170.00} in

In [40]:
validate.summary()

Access to fu T0: 4 of 10 fit | Locked out of account or stuck in verification
Access to fu T2: 7 of 10 fit | Check deposit held, funds not available
Access to fu T4: 7 of 10 fit | Account closed or funds hard to withdraw (tentative, catch-all cluster)
Debits and p T3: 6 of 10 fit | Sign-up bonus or direct deposit promotion not honored
Disputes and T3: 7 of 10 fit | Charge disputed with a company after payment
Disputes and T5: 7 of 10 fit | Unauthorized debit card transactions, claim denied
Disputes and T6: 7 of 10 fit | Money taken from a checking account or ATM, no help
Disputes and T7: 8 of 10 fit | Merchant refund or non-delivery dispute
Fraud and sc T5: 5 of 10 fit | Fake check or job scam
Unclassified T3: 9 of 10 fit | Wire transfer problem, recall requested
